# 量子テレポーテーション (宿題)

## はじめに・準備

初日の講義でも紹介した１量子ビットを送るための量子テレポーテーションをpytketをつかって実装してみましょう。
量子テレポーテーションでは伝統的に、量子状態を送る人をAlice、受け取る人をBobと呼ぶため、ここでもそれに従っています。

In [1]:
!pip install pytket-offline-display

  Using cached pytket_offline_display-0.0.9-py3-none-any.whl.metadata (1.6 kB)
Using cached pytket_offline_display-0.0.9-py3-none-any.whl (202 kB)


講義で説明したように量子テレポーテーションの回路は以下のように描くことが出来ます。


<img src="./fig/quantum_teleportation_circ.png" width="750">


これを再現してみましょう！

### 回路を作るために必要なモジュールをインポートしましょう。

In [2]:
from pytket import Circuit #回路を作成するためのもの
from pytket.circuit.display import render_circuit_jupyter # 回路を図示するためのもの
from pytket.extensions.offline_display import get_circuit_renderer

### 回路の定義
まずは量子ビットと古典ビットから成る回路を定義します。

In [3]:
circ = Circuit(3, 2)  # 3量子ビット、2古典ビット
# 実際に描いてみると...
circuit_renderer = get_circuit_renderer()
circuit_renderer.render_circuit_jupyter(circ)

講義でも触れたようにこれは量子ビットと古典ビットを準備しただけで何もゲートが含まれていない空の回路です。２日目のハンズオンでも紹介しましたが、本来は
```circ = Circuit()
alice = circ.add_q_register("a", 2)
bob = circ.add_q_register("b", 1)
cr = circ.add_c_register("c", 2)
```
のように、それぞれの量子ビットに名前を付けた方が分かりやすいです。しかし、シンプルさを重視してここでは省略します。

レジスタに名前を付けなかったため、この図だけからは判断できませんがAliceが上２つの量子ビット、Bobが一番下の量子ビットを持っている、という設定です。
このうち、一番上の量子ビットが送りたい状態、下２つがそれを可能にするためにAliceとBobでひとつずつ持つ量子ビットです。

※上で準備された回路では３つの量子ビットが全て$\ket{0}$に初期化されています。次のセルではAliceが送りたい状態をここでは人工的に作るための操作です。少し複雑ですが、ランダムに状態を作っている、ということだけ理解していれば問題ありません。具体的には初期の$\ket{0}$を２つのランダムの角度によるブロッホ球面上での回転でランダムな状態にしているだけです。
(ここでは、必ずしも必要というわけではありませんが2日目に説明した回路のbox化を利用しています。)

In [4]:
# ２つのランダムな角度の生成
import numpy as np
from pytket.circuit import CircBox
theta = np.random.uniform(0, np.pi)  # [0, π] の範囲でランダムなθ
phi = np.random.uniform(0, 2 * np.pi)  # [0, 2π] の範囲でランダムなφ

sub = Circuit(1) # Aliceの持つ初期状態を作るためのパーツをboxとして準備
sub.name = 'Initial state'
sub.Ry(theta, 0).Rz(phi, 0) # Ry(θ) と Rz(φ) を適用(Bloch 球のパラメータ化)
sub_box = CircBox(sub)

circ.add_circbox(sub_box, [0]) #作ったboxを先の回路に追加
render_circuit_jupyter(circ)

以下の解説ではこの状態を$\ket{\psi}=a\ket{0}+b\ket{1}$とします。

## 量子テレポーテーションの回路を作ってみる (以下は解説なので、挑戦してみたい人は自力で考えてみてください)

ここからが量子テレポーテーションの回路の作成です。(上のセルで$\ket{\psi}$をAliceに持たせるために0番目の量子ビットに作用させたゲートを除けば空の回路です。)

まず、下の２つの量子ビットの間に量子力学的な相関(エンタングルメント)を持たせるためにこれらをベル状態(Bell state)にします。

In [5]:
circ.H(1).CX(1,2)
#これも実際に描いてみましょう
render_circuit_jupyter(circ)

次にAliceの手元でベル測定をします。講義でも紹介したように測定はある決められた正規直交基底で行ないますが、この場合は２量子ビットなので４つの状態からなる基底になります:
$$ \frac{1}{\sqrt{2}} (\ket{00}+\ket{11}) $$
$$ \frac{1}{\sqrt{2}} (\ket{01}+\ket{10}) $$
$$ \frac{1}{\sqrt{2}} (\ket{00}-\ket{11}) $$
$$ \frac{1}{\sqrt{2}} (\ket{01}-\ket{10}) $$

しかし、実際に量子コンピュータで行う測定自体はパウリ$Z$(今の場合、２量子ビットを考えているため正確には$Z_0 \otimes Z_1$)の固有ベクトルの基底$\{ \ket{00}, \ket{01}, \ket{10}, \ket{11} \}$で行います。そのため、Aliceは測定の前に手元の２量子ビットに対して操作する必要がある、というのが右上の測定マークの手前の操作の意味です。この部分を回路に追加してみましょう

In [6]:
circ.CX(0,1).H(0)
#これも実際に描いてみましょう
#render_circuit_jupyter(circ)

[CircBox q[0]; H q[1]; CX q[1], q[2]; CX q[0], q[1]; H q[0]; ]

In [7]:
circ.Measure(0,0).Measure(1,1)
#これも実際に描いてみましょう
render_circuit_jupyter(circ)

ここで ```circ.Measure(0,0)```は0番目の量子ビットの測定結果を0番目の古典ビットに格納する、という意味です。Aliceが持つもう１つの量子ビットの測定もしたいので続けて```.Measure(1,1)``` としています。このように1行でユニタリゲートを複数追加できたように測定も連続して追加できます。測定の結果、次の４種類の２古典ビット$\{00,01,10,11\}$が「等確率で」得られます。
(等確率である理由を考えてみましょう！)

実はAliceが手元の状態を「測定した瞬間に」Bobの手元の状態も「Aliceの測定結果に依存して」以下の４通りに射影されます。AliceとBobがどれだけ離れていても関係ありません。これがエンタングルメントの面白いところです。(勘のいい人はこれは相対論に反するのでは？と思うかもしれません。このカラクリはこの後までみると分かります。)

$$ a\ket{0}+b\ket{1} $$
$$ a\ket{1}+b\ket{0} $$
$$ a\ket{0}-b\ket{1} $$
$$ a\ket{1}- b\ket{0} $$

この順番はAliceの得うる$\{00,01,10,11\}$の順番です。Bobに最終的に送りたいのは$\ket{\psi}=a\ket{0}+b\ket{1}$です。つまり、$\ket{\psi}$を得るためにBobは手元で測定結果に応じた操作をします。そのために、上の４つの状態と$\ket{\psi}$の関係を見ておきましょう。

$$ a\ket{0}+b\ket{1} = \ket{\psi}$$
$$ a\ket{1}+b\ket{0} = X \ket{\psi}$$
$$ a\ket{0}-b\ket{1} = Z \ket{\psi}$$
$$ a\ket{1}- b\ket{0} = XZ \ket{\psi}$$

つまり、Aliceの手元の古典ビットに格納された測定結果 $pq~(p,q=0,1)$ に対してBobが持っている状態は
$$ X^q Z^p \ket{\psi} $$
だと分かります！

つまりBobの手元で自分が持っている状態に$Z^p X^q$を作用させれば、無事Bobのもとに$\ket{\psi}$が送られたことになります。では、Aliceの測定結果に応じてBobの手元の状態にユニタリゲートを作用させましょう。

In [8]:
circ.X(2, condition_bits=[1])
circ.Z(2, condition_bits=[0])

[CircBox q[0]; H q[1]; CX q[1], q[2]; CX q[0], q[1]; Measure q[1] --> c[1]; H q[0]; Measure q[0] --> c[0]; IF ([c[1]] == 1) THEN X q[2]; IF ([c[0]] == 1) THEN Z q[2]; ]

このように古典ビットの値を条件とした(古典ビットを制御ビットとした)ユニタリゲートの作用のさせ方もできます。

In [9]:
render_circuit_jupyter(circ)

これで量子テレポーテーションの回路が描けたことになります。

## 発展

本来は
```
circ = Circuit()
alice = circ.add_q_register("a", 2)
bob = circ.add_q_register("b", 1)
cr = circ.add_c_register("c", 2)
```
のように、それぞれの量子ビットに名前を付けて回路を初期化した方が分かりやすい、と２日目のハンズオンおよび、このノートブックの最初に述べました。上のシンプルな方法で実装出来たら、今度はこれでもう一度量子テレポーテーションの回路を作ってみましょう！
答えは[こちら](https://docs.quantinuum.com/tket/user-guide/examples/circuit_construction/conditional_gate_example.html)
のページにあります。

## おわりに

このノートブックでは、量子テレポーテーションの回路を実際に手を動かして描いてもらいました。量子テレポーテーションは量子情報理論的な側面での重要性もさることながら、量子計算をするためのプログラミングの題材としても教育的なものです。回路の定義や１量子あるいは２量子ゲート操作といった初歩的な操作や、中間測定とその結果を利用したゲート操作を含みます。ここでは必ずしも必要ではありませんでしたが、回路の一部のbox化は三日目の演習で行なうような、より複雑な回路を作成する上で重要です。

## 参考文献

Michael A. Nielsen, Isaac L. Chuang, [*Quantum Computation and Quantum Information*](https://www.cambridge.org/highereducation/books/quantum-computation-and-quantum-information/01E10196D0A682A6AEFFEA52D53BE9AE#overview)